In [ ]:
def visualise_temperature(u, interior_mask, bid=None, save_path=None, show=True):
    old_u = None
    old_mask = None
    if bid:
        old_u, old_mask = load_data(LOAD_DIR, bid)

    padded_mask = np.zeros_like(u, dtype=bool)
    padded_mask[1:-1, 1:-1] = interior_mask

    interior_only = np.ma.masked_where(~padded_mask, u)
    walls = np.zeros_like(u, dtype=int)
    walls[u == 5] = 1
    walls[u == 25] = 2
    walls = np.ma.masked_where(walls == 0, walls)

    fig, axes = plt.subplots(1, 2, figsize=(16, 5), constrained_layout=True)

    if old_u is not None and old_mask is not None:
        old_padded_mask = np.zeros_like(old_u, dtype=bool)
        old_padded_mask[1:-1, 1:-1] = old_mask
        old_categories = np.zeros_like(old_u, dtype=int)
        old_categories[old_u == 5] = 1
        old_categories[old_u == 25] = 2
        old_categories[old_padded_mask] = 3

        floorplan_cmap = ListedColormap(["white", "#4c78a8", "#f58518", "#54a24b"])
        floorplan_norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5], floorplan_cmap.N)
        floorplan_plot = axes[0].imshow(old_categories, origin="lower", cmap=floorplan_cmap, norm=floorplan_norm)
        axes[0].set_title(f"Floor Plan: {bid}")
        axes[0].set_xlabel("x")
        axes[0].set_ylabel("y")
        colorbar = fig.colorbar(floorplan_plot, ax=axes[0], shrink=0.85, ticks=[0, 1, 2, 3])
        colorbar.ax.set_yticklabels(["outside", "load-bearing wall", "inside wall", "interior"])
    else:
        axes[0].axis("off")

    overlay_plot = axes[1].imshow(interior_only, origin="lower", cmap="coolwarm")
    wall_cmap = ListedColormap(["#4c78a8", "#f58518"])
    wall_norm = BoundaryNorm([0.5, 1.5, 2.5], wall_cmap.N)
    axes[1].imshow(walls, origin="lower", cmap=wall_cmap, norm=wall_norm, alpha=0.95)
    axes[1].set_title(f"Temperature of {bid}")
    axes[1].set_xlabel("x")
    axes[1].set_ylabel("y")
    fig.colorbar(overlay_plot, ax=axes[1], shrink=0.85, label="Temperature")

    if save_path is not None:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")

    if show:
        plt.show()
    else:
        plt.close(fig)

    return fig, axes

In [ ]:
def visualise_speedup(speedup_data, save_path=None, show=True):
    fig, ax = plt.subplots(figsize=(8, 6), constrained_layout=True)
    for label, group in speedup_data.groupby("Scheduling"):
        ax.plot(group["Cores"], group["Time Taken"], marker="o", label=label)
    ax.set_title("Speedup vs Number of Cores")
    ax.set_xlabel("Number of Cores")
    ax.set_ylabel("Time Taken (s)")
    ax.set_xticks(speedup_data["Time Taken"].unique())
    ax.legend()
    ax.grid(True)

    if save_path is not None:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")

    if show:
        plt.show()
    else:
        plt.close(fig)